In [1]:
import os
import re
import fitz
import numpy as np

from sentence_transformers import SentenceTransformer
from pymongo import MongoClient, ReplaceOne

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PDF_PATH = r"./BNS_IPC_Comparative.pdf"

DB_NAME = "test_db"
COLLECTION_NAME = "mapping"

NEW_LAW = "BNS"
OLD_LAW = "IPC"

VECTOR_INDEX_NAME = "test_mapping"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

MAX_CHUNK_TOKENS = 220

In [3]:
pdf = fitz.open(PDF_PATH)

print("Total pages:", len(pdf))

Total pages: 23


In [4]:
def clean_cell(text):

    if not text:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def expand_section_ref(ref):

    match = re.match(
        r"^(\d{1,3}[A-Z]{0,3})(.*)$",
        ref
    )

    if not match:
        return [ref]

    base, tail = match.groups()

    groups = re.findall(
        r"\(([^)]+)\)",
        tail
    )

    result = [base]

    # Handle grouped subsections like 309(1)(2)(3)
    if (
        len(groups) > 1
        and all(g.isdigit() for g in groups)
    ):

        for group in groups:

            result.append(
                f"{base}({group})"
            )

    else:

        current = base

        for group in groups:

            current += f"({group})"

            result.append(current)

    return list(dict.fromkeys(result))

In [5]:
REF_START_RE = re.compile(

    r"^\s*"

    r"(\d{1,3}[A-Za-z]{0,3}"

    r"(?:\s*\(\s*(?:\d+|[A-Za-z])\s*\))*"

    r"(?:\s*,\s*\d{1,3}[A-Za-z]{0,3}"

    r"(?:\s*\(\s*(?:\d+|[A-Za-z])\s*\))*)*)"

    r"\s*(?:[.)-]\s*|\s+|$)"
)

In [6]:
def extract_section_refs(text):

    refs = []

    for line in (text or "").splitlines():

        line = line.strip()

        if not line:
            continue

        match = REF_START_RE.match(line)

        if match:

            raw_refs = match.group(1)

            individual_refs = re.split(
                r",\s*(?=\d)",
                raw_refs
            )

            for raw in individual_refs:

                normalized = re.sub(
                    r"\s+",
                    "",
                    raw
                ).upper()

                for ref in expand_section_ref(
                    normalized
                ):

                    if ref not in refs:
                        refs.append(ref)

        # Special case: Explanation to 489A
        for match in re.finditer(

            r"\bexplanation\s+to\s+"
            r"(\d{1,3}[A-Za-z]{0,3})\b",

            line,

            re.IGNORECASE
        ):

            ref = match.group(1).upper()

            if ref not in refs:
                refs.append(ref)

    return refs

In [7]:
print(extract_section_refs("351 (2), 351 (3)"))

print(extract_section_refs("125(a)"))

print(extract_section_refs("376AB. Punishment for rape"))

print(extract_section_refs("309 (1)(2)(3)"))

['351', '351(2)', '351(3)']
['125', '125(A)']
['376AB']
['309', '309(1)', '309(2)', '309(3)']


In [8]:
def is_header_row(left, right):

    text = (
        left + " " + right
    ).lower()

    if (
        "bharatiya nyaya sanhita" in text
        and "indian penal code" in text
    ):
        return True

    if (
        "2023 (bns)" in text
        and "1860 (ipc)" in text
    ):
        return True

    return False

In [9]:
def is_heading_row(left, right):

    text = (
        left + " " + right
    ).strip()

    return bool(
        re.match(
            r"^(?:CHAPTER\b|Chapter\b|Of\b|Sexual Offences\b)",
            text
        )
    )

In [10]:
def extract_mapping_rows(pdf):

    records = []

    for page_number, page in enumerate(
        pdf,
        start=1
    ):

        tables = page.find_tables()

        candidates = []

        for table in tables.tables:

            rows = table.extract()

            if (
                rows
                and max(len(row) for row in rows) == 2
                and len(rows) > 5
            ):

                candidates.append(rows)

        if not candidates:
            continue

        # Select the main two-column comparison table
        rows = max(
            candidates,
            key=len
        )

        for raw_row in rows:

            left = clean_cell(
                raw_row[0] or ""
            )

            right = clean_cell(
                raw_row[1] or ""
            )

            if is_header_row(left, right):
                continue

            if not left and not right:
                continue

            new_refs = extract_section_refs(left)

            old_refs = extract_section_refs(right)

            deleted = bool(
                re.search(
                    r"\bdeleted\b",
                    left,
                    re.IGNORECASE
                )
            )

            new_marker = bool(
                re.search(
                    r"new\s+(?:sub\s*-?\s*)?section",
                    right,
                    re.IGNORECASE
                )
            )

            # --------------------------------
            # Continuation of old section
            # --------------------------------

            if (
                not left
                and (old_refs or new_marker)
                and records
            ):

                records[-1]["old_raw"] += (
                    "\n" + right
                )

                records[-1]["old_sections"] = list(
                    dict.fromkeys(
                        records[-1]["old_sections"]
                        + old_refs
                    )
                )

                records[-1]["page_end"] = page_number

                continue

            # --------------------------------
            # New mapping record
            # --------------------------------

            if (
                deleted
                or new_refs
                or old_refs
                or new_marker
            ):

                records.append({

                    "new_raw": left,

                    "old_raw": right,

                    "new_sections": new_refs,

                    "old_sections": old_refs,

                    "page_start": page_number,

                    "page_end": page_number
                })

            # --------------------------------
            # Continuation of existing text
            # --------------------------------

            elif not is_heading_row(left, right):

                if records:

                    if left:

                        records[-1]["new_raw"] += (
                            "\n" + left
                        )

                    if right:

                        records[-1]["old_raw"] += (
                            "\n" + right
                        )

                    records[-1]["page_end"] = page_number

    return records

In [11]:
raw_rows = extract_mapping_rows(pdf)

print(
    "Extracted mapping rows:",
    len(raw_rows)
)

Extracted mapping rows: 554


In [12]:
mapping_rows = []

for index, row in enumerate(raw_rows):

    new_sections = row["new_sections"]

    old_sections = row["old_sections"]

    new_raw = row["new_raw"]

    old_raw = row["old_raw"]

    change_flag = bool(
        re.search(
            r"\(\s*Change\s*\)",
            new_raw,
            re.IGNORECASE
        )
    )

    deleted_flag = bool(
        re.search(
            r"\bDeleted\b",
            new_raw,
            re.IGNORECASE
        )
    )

    new_subsection_flag = bool(
        re.search(
            r"New\s+Sub\s*-?\s*Section",
            old_raw,
            re.IGNORECASE
        )
    )

    new_section_flag = bool(
        re.search(
            r"New\s+Section",
            old_raw,
            re.IGNORECASE
        )
    )

    # Classification
    if deleted_flag:

        mapping_type = "deleted_in_new_law"

    elif new_subsection_flag:

        mapping_type = "new_sub_section"

    elif new_section_flag:

        mapping_type = "new_section"

    elif change_flag:

        mapping_type = "marked_change"

    elif new_sections and old_sections:

        mapping_type = "listed_correspondence"

    elif new_sections:

        mapping_type = "no_old_reference_listed"

    else:

        mapping_type = "old_reference_without_new_section"


    mapping_rows.append({

        "mapping_id":
            f"bns-ipc-{index + 1:04d}",

        "new_law": "BNS",

        "old_law": "IPC",

        "new_section": (
            new_sections[0] if new_sections else ""
        ),

        "old_section": (
            old_sections[0] if old_sections else ""
        ),

        "new_sections": new_sections,

        "old_sections": old_sections,

        "new_section_raw": new_raw,

        "old_section_raw": old_raw,

        "mapping_type": mapping_type,

        "change_flag": change_flag,

        "page_start": row["page_start"],

        "page_end": row["page_end"],

        "document_type": "law_mapping",

        "source": os.path.basename(PDF_PATH)

    })

In [13]:
def inspect_bns_mapping(section):

    section = str(section).upper()

    results = [

        row for row in mapping_rows

        if section in row["new_sections"]

    ]

    for row in results:

        print("\nBNS:", row["new_sections"])

        print("IPC:", row["old_sections"])

        print("Type:", row["mapping_type"])

        print("BNS entry:", row["new_section_raw"])

        print("IPC entry:", row["old_section_raw"])

        print("-" * 70)

In [14]:
inspect_bns_mapping("103")

inspect_bns_mapping("111")

inspect_bns_mapping("178")

inspect_bns_mapping("325")

inspect_bns_mapping("351(2)")


BNS: ['103']
IPC: ['302']
Type: marked_change
BNS entry: 103 Punishmentformurder (Change) . .
IPC entry: 302 Punishmentformurder . .
----------------------------------------------------------------------

BNS: ['111']
IPC: []
Type: no_old_reference_listed
BNS entry: 111 Organisedcrime .
IPC entry: NewSection
----------------------------------------------------------------------

BNS: ['178']
IPC: ['231']
Type: listed_correspondence
BNS entry: 178 Counterfeiting coin Government stamps . , , currency-notes or bank -notes .
Explanation
IPC entry: 231 Counterfeitingcoin . . 255 CounterfeitingGovernmentstamp . . 489A Counterfeiting currency-notes or bank -notes . .
----------------------------------------------------------------------

BNS: ['178', '178(1)']
IPC: []
Type: no_old_reference_listed
BNS entry: 178(1)
IPC entry: Explanationto489A
----------------------------------------------------------------------

BNS: ['178', '178(2)']
IPC: ['230']
Type: listed_correspondence
BNS entry: 178

In [15]:
def inspect_ipc_mapping(section):

    section = str(section).upper()

    for row in mapping_rows:

        if section in row["old_sections"]:

            print(
                "IPC:",
                row["old_sections"]
            )

            print(
                "BNS:",
                row["new_sections"]
            )

            print(
                "Type:",
                row["mapping_type"]
            )

            print(
                "Entry:",
                row["old_section_raw"]
            )

            print("-" * 60)

In [16]:
inspect_ipc_mapping("124A")

IPC: ['124A']
BNS: []
Type: deleted_in_new_law
Entry: 124A Sedition .
------------------------------------------------------------


In [17]:
model = SentenceTransformer(
    EMBEDDING_MODEL
)

print(
    "Embedding dimensions:",
    model.get_sentence_embedding_dimension()
)

print(
    "Maximum sequence length:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4071.18it/s]


Embedding dimensions: 384
Maximum sequence length: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_17852\2227726617.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [18]:
def clean_cell(text):

    if not text:
        return ""

    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in str(text).splitlines()
    ]

    return "\n".join(
        line for line in lines if line
    )

In [19]:
def token_count(text):

    return len(
        model.tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )

In [20]:
def split_by_tokens(
    text,
    max_tokens,
    overlap=10
):

    token_ids = model.tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    if len(token_ids) <= max_tokens:
        return [text]

    chunks = []

    step = max_tokens - overlap

    if step <= 0:
        raise ValueError("Invalid token overlap")

    for start in range(
        0,
        len(token_ids),
        step
    ):

        part_ids = token_ids[
            start:start + max_tokens
        ]

        part = model.tokenizer.decode(
            part_ids,
            skip_special_tokens=True
        )

        chunks.append(part)

        if start + max_tokens >= len(token_ids):
            break

    return chunks

In [21]:
def create_mapping_chunks(mapping_rows):

    chunks = []

    for row in mapping_rows:

        new_refs = (
            ", ".join(row["new_sections"])
            if row["new_sections"]
            else "No BNS section listed"
        )

        old_refs = (
            ", ".join(row["old_sections"])
            if row["old_sections"]
            else "No IPC section listed"
        )

        header = f"""
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: {new_refs}

IPC Sections: {old_refs}

Mapping Type: {row['mapping_type']}
""".strip()

        body = f"""
BNS Table Entry:

{row['new_section_raw'] or 'Deleted / not listed'}

IPC Table Entry:

{row['old_section_raw'] or 'No corresponding IPC entry listed'}
""".strip()

        full_text = header + "\n\n" + body

        # Keep normal mapping rows intact
        if token_count(full_text) <= MAX_CHUNK_TOKENS:

            parts = [body]

        else:

            available_tokens = (
                MAX_CHUNK_TOKENS
                - token_count(header)
                - 12
            )

            while available_tokens > 20:

                parts = split_by_tokens(
                    body,
                    max_tokens=available_tokens,
                    overlap=10
                )

                test_texts = [
                    header + "\n\n" + part
                    for part in parts
                ]

                if all(
                    token_count(text) <= MAX_CHUNK_TOKENS
                    for text in test_texts
                ):
                    break

                available_tokens -= 5

            else:

                raise ValueError(
                    f"Cannot split mapping: "
                    f"{row['mapping_id']}"
                )

        for index, part in enumerate(parts):

            chunk_text = (
                header + "\n\n" + part
            ).strip()

            if token_count(chunk_text) > MAX_CHUNK_TOKENS:

                raise ValueError(
                    f"Token limit exceeded: "
                    f"{row['mapping_id']}"
                )

            chunks.append({

                "text": chunk_text,

                "metadata": {

                    **row,

                    "chunk_index": index

                }

            })

    return chunks

In [22]:
chunks = create_mapping_chunks(
    mapping_rows
)

print("Total mapping rows:", len(mapping_rows))

print("Total final chunks:", len(chunks))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (264 > 256). Running this sequence through the model will result in indexing errors


Total mapping rows: 554
Total final chunks: 555


In [23]:
invalid_chunks = []

max_tokens = 0

for chunk in chunks:

    count = token_count(
        chunk["text"]
    )

    max_tokens = max(
        max_tokens,
        count
    )

    if count > model.max_seq_length:

        invalid_chunks.append({

            "mapping_id":
                chunk["metadata"]["mapping_id"],

            "tokens": count
        })


print("Total chunks:", len(chunks))

print("Maximum tokens:", max_tokens)

print("Invalid chunks:", len(invalid_chunks))

print("Invalid examples:", invalid_chunks[:5])


assert len(invalid_chunks) == 0

Total chunks: 555
Maximum tokens: 208
Invalid chunks: 0
Invalid examples: []


In [24]:
embedding_texts = [

    chunk["text"]

    for chunk in chunks
]


embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)


print("Embedding shape:", embeddings.shape)

assert embeddings.shape[1] == 384

assert len(embeddings) == len(chunks)

Batches: 100%|██████████| 18/18 [00:09<00:00,  1.90it/s]

Embedding shape: (555, 384)


In [25]:
MONGO_URI = os.getenv("MONGODB_URI")

client = MongoClient(MONGO_URI)

db = client[DB_NAME]

collection = db[COLLECTION_NAME]


print(
    client.admin.command("ping")
)

{'ok': 1}


In [26]:
collection.create_index(
    "id",
    unique=True
)


collection.create_index([

    ("metadata.new_law", 1),

    ("metadata.new_sections", 1)

])


collection.create_index([

    ("metadata.old_law", 1),

    ("metadata.old_sections", 1)

])

'metadata.old_law_1_metadata.old_sections_1'

In [27]:
mongo_documents = []

for i, chunk in enumerate(chunks):

    metadata = chunk["metadata"].copy()

    metadata["text"] = chunk["text"]

    mongo_documents.append({

        "id": (
            f"{metadata['mapping_id']}-"
            f"{metadata['chunk_index']}"
        ),

        "values": embeddings[i].tolist(),

        "metadata": metadata

    })


print(
    "Prepared documents:",
    len(mongo_documents)
)

Prepared documents: 555


In [28]:
operations = [

    ReplaceOne(

        {"id": doc["id"]},

        doc,

        upsert=True

    )

    for doc in mongo_documents
]


if operations:

    result = collection.bulk_write(
        operations,
        ordered=False
    )

    print(
        "Inserted:",
        result.upserted_count
    )

    print(
        "Modified:",
        result.modified_count
    )

Inserted: 555
Modified: 0


In [29]:
print(
    "BNS-IPC records:",

    collection.count_documents({

        "metadata.new_law": "BNS",

        "metadata.old_law": "IPC"

    })
)


print(
    "Total mapping collection records:",

    collection.count_documents({})
)

BNS-IPC records: 555
Total mapping collection records: 1301


In [30]:
def mongo_mapping_search(query, k=5):

    query_vector = model.encode(

        [query],

        normalize_embeddings=True

    )[0].tolist()


    pipeline = [

        {
            "$vectorSearch": {

                "index": VECTOR_INDEX_NAME,

                "path": "values",

                "queryVector": query_vector,

                "numCandidates": max(100, k * 20),

                "limit": k,

                "filter": {

                    "metadata.new_law": "BNS",

                    "metadata.old_law": "IPC"

                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {

                    "$meta": "vectorSearchScore"

                }
            }
        }
    ]


    return list(
        collection.aggregate(pipeline)
    )

In [31]:
results = mongo_mapping_search(

    "What is the IPC equivalent of BNS Section 103?",

    k=5
)


for result in results:

    print("\nScore:", result["score"])

    print(
        "BNS:",
        result["metadata"]["new_sections"]
    )

    print(
        "IPC:",
        result["metadata"]["old_sections"]
    )

    print(
        result["metadata"]["text"]
    )

    print("=" * 70)


Score: 0.8639335036277771
BNS: ['317', '317(5)']
IPC: ['414']
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: 317, 317(5)

IPC Sections: 414

Mapping Type: listed_correspondence

BNS Table Entry:

317(5)
Ofcheating

IPC Table Entry:

414 Assistinginconcealmentofstolenproperty . .
OfCheating

Score: 0.8638004064559937
BNS: ['199']
IPC: ['166A']
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: 199

IPC Sections: 166A

Mapping Type: listed_correspondence

BNS Table Entry:

199 Publicservantdisobeyingdirectionunderlaw . .

IPC Table Entry:

166A Publicservantdisobeyingdirectionunderlaw . .

Score: 0.8618881702423096
BNS: ['344']
IPC: ['477A']
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: 344

IPC Sections: 477A

Mapping Type: listed_correspondence

BNS Table Entry:

344 Falsificationofaccounts . .
Ofpropertymarks

IPC Table Entry:

477A Falsificationofaccounts . .
OfPropertyandOtherMarks

Score: 0.8611072897911072
BNS: ['310', '310(5)']
IPC: ['402']
Law Mapping: BNS 2023 to IPC 1860



In [32]:
def find_bns_section(section):

    section = re.sub(
        r"\s+",
        "",
        str(section)
    ).upper()

    query = {

        "metadata.new_law": "BNS",

        "metadata.old_law": "IPC",

        "metadata.new_sections": section

    }

    results = list(
        collection.find(
            query,
            {"_id": 0}
        )
    )

    # If a subsection has no separate IPC mapping,
    # retrieve its parent section correspondence.
    if (
        not any(
            doc["metadata"].get("old_sections")
            for doc in results
        )
        and "(" in section
    ):

        parent = re.match(
            r"^\d+[A-Z]{0,3}",
            section
        ).group()

        parent_results = list(
            collection.find(

                {
                    "metadata.new_law": "BNS",

                    "metadata.old_law": "IPC",

                    "metadata.new_sections": parent
                },

                {"_id": 0}
            )
        )

        results.extend(parent_results)

    return results

In [33]:
def find_ipc_section(section):

    section = re.sub(
        r"\s+",
        "",
        str(section)
    ).upper()

    return list(
        collection.find(

            {

                "metadata.new_law": "BNS",

                "metadata.old_law": "IPC",

                "metadata.old_sections": section

            },

            {"_id": 0}

        )
    )

In [34]:
# BNS Section 103 -> IPC

results = find_bns_section("103")

for result in results:

    print(
        "BNS:",
        result["metadata"]["new_sections"]
    )

    print(
        "IPC:",
        result["metadata"]["old_sections"]
    )

    print(
        result["metadata"]["text"]
    )

    print("-" * 60)

BNS: ['103']
IPC: ['302']
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: 103

IPC Sections: 302

Mapping Type: marked_change

BNS Table Entry:

103 Punishmentformurder (Change) . .

IPC Table Entry:

302 Punishmentformurder . .
------------------------------------------------------------


In [35]:
# IPC Section 302 -> BNS

results = find_ipc_section("302")

for result in results:

    print(
        "IPC:",
        result["metadata"]["old_sections"]
    )

    print(
        "BNS:",
        result["metadata"]["new_sections"]
    )

    print(
        result["metadata"]["text"]
    )

IPC: ['302']
BNS: ['103']
Law Mapping: BNS 2023 to IPC 1860

BNS Sections: 103

IPC Sections: 302

Mapping Type: marked_change

BNS Table Entry:

103 Punishmentformurder (Change) . .

IPC Table Entry:

302 Punishmentformurder . .


In [36]:
results = find_bns_section("351(2)")

for result in results:

    print(
        "BNS:",
        result["metadata"]["new_sections"]
    )

    print(
        "IPC:",
        result["metadata"]["old_sections"]
    )

BNS: ['351', '351(2)']
IPC: ['506']
